# World of Shadow Work — vessel splats via **TRELLIS** (Colab, Python 3.10 / condacolab)

Generates **128** high-quality, **dense** (`G=30000`) image-to-3D Gaussian-splat keyframes for the runtime's morphing vessel, with **TRELLIS** (Microsoft image-to-3D structured-latent — top 3DGS quality). One splat per input image; the runtime melts between them.

**Why condacolab:** TRELLIS needs **Python 3.10** — its `spconv`/`cumm` sparse stack has no 3.12 wheel and won't build from source. `condacolab` gives Colab a 3.10 base where `spconv-cu118` installs as a **real** wheel.

**RUN ORDER (important):**
1. Run **Cell 1 (condacolab)** first — it installs Python 3.10 and **restarts the kernel once**.
2. When it reconnects, **Runtime → Run all** again. Cell 1 becomes a no-op and it proceeds: install → corpus → TRELLIS → pack → download.

**Runtime:** A100 (80GB ideal). Set Runtime → Change runtime type → **A100 GPU** BEFORE starting.

Outputs `assets/vessel.wswv` and **downloads it to your Mac** (never git-push). `DENS=4` is already set in `viz/VesselSplats.cpp` for these dense clouds — on the Mac: `cp ~/Downloads/vessel.wswv reagency/assets/vessel.wswv`.

## Cell 1 · condacolab — Python 3.10 base (RESTARTS the kernel once; then Run-all again)

In [ ]:
# Installs a Python-3.10 conda base and RESTARTS once. On re-run it's a no-op (check passes) and flow continues.
try:
    import condacolab; condacolab.check()
    print('condacolab ready (Python 3.10 base)')
except Exception:
    !pip install -q condacolab
    import condacolab; condacolab.install()   # <-- RESTARTS the kernel here; when it reconnects, Run all again

## Cell 2 · Config (one source of truth)

In [ ]:
REPO     = '9LiveZZZ-Git/MAT201B_Projects'
BRANCH   = 'world-of-shadow-work'   # matches wosw_dream_colab (the path that worked)

# Corpus from Google Drive (AIC source URLs 403 on direct fetch — same reuse as wosw_dream_colab).
USE_DRIVE_CORPUS = True
DRIVE_CORPUS     = '/content/drive/MyDrive/wosw/corpus.zip'   # a .zip OR a folder on your Drive

N_IMAGES = 128      # splat keyframes (one per input image)
SELECT   = 'spread' # corpus images: 'spread' (evenly across corpus) | 'first' (incl. the dream sources)
STEPS    = 50       # TRELLIS sampling steps (sparse-structure + SLAT); 50 = high quality
G        = 30000    # gaussians/keyframe (pairs with runtime DENS=4)
MAX_MB   = 80       # vessel.wswv size cap (128 x 30000 x 16 B ~= 61 MB; keep above that so G holds)
print('config:', N_IMAGES, 'keyframes | G', G, '| STEPS', STEPS, '| corpus:', DRIVE_CORPUS)

## Cell 3 · GPU + Python check (expect Python 3.10.x + an A100)

In [ ]:
import sys; print('python', sys.version.split()[0])           # expect 3.10.x (condacolab)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## Cell 4 · Install TRELLIS  (Py3.10 + Torch 2.1/cu118 + REAL spconv-cu118)
TRELLIS's supported combo. `spconv-cu118` + `kaolin` both have 3.10 wheels here, so **no mocks**. ~10–15 min the first run. (`flash-attn` omitted on purpose — `xformers` covers attention and avoids a long compile.)

In [ ]:
import os, importlib
os.chdir('/content')
if not os.path.isdir('/content/TRELLIS'):
    !git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git
# Torch 2.1.0 + CUDA 11.8 (Colab's 12.x driver runs cu118 fine); then rembg-compatible Pillow + io deps.
!pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118
!pip install -q "pillow>=12.1,<13" rembg onnxruntime plyfile imageio imageio-ffmpeg
os.chdir('/content/TRELLIS')
# Gaussian path needs: basic + xformers (attn) + spconv (sparse backend) + the gaussian rasterizer + kaolin.
# If from_pretrained later complains about a missing renderer, add: --nvdiffrast --diffoctreerast
!. ./setup.sh --basic --xformers --spconv --mipgaussian --kaolin
importlib.import_module('spconv.pytorch'); print('spconv OK (REAL wheel) — TRELLIS ready')

## Cell 5 · Repo + corpus inputs from Drive  ->  `work/vessel_inputs/`
Clones the project, mounts Drive, restores the corpus from `DRIVE_CORPUS` (zip -> unzip, folder -> symlink), and copies `N_IMAGES` of `corpus/images/**/*.jpg` in (sorted = the `stage_a_embed.py` node order). Reuses existing inputs if already populated.

In [ ]:
import os, glob, shutil
os.chdir('/content')
if not os.path.isdir('/content/MAT201B_Projects'):
    !git clone -b {BRANCH} https://github.com/{REPO}.git
FAC = '/content/MAT201B_Projects/reagency/factory'
os.chdir(FAC)
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('work/vessel_inputs', exist_ok=True); os.makedirs('work/vessels', exist_ok=True)

INP = 'work/vessel_inputs'
existing = sorted(glob.glob(f'{INP}/*.jpg') + glob.glob(f'{INP}/*.png'))
if len(existing) >= N_IMAGES:
    print(f'reusing {len(existing)} existing inputs in {INP}/')
else:
    if USE_DRIVE_CORPUS and not os.path.isdir('corpus/images'):
        !rm -rf corpus _cz
        if DRIVE_CORPUS.endswith('.zip'):
            !unzip -q -o "{DRIVE_CORPUS}" -d _cz
            shutil.move('_cz/corpus' if os.path.isdir('_cz/corpus') else '_cz', 'corpus')
        else:
            os.symlink(DRIVE_CORPUS, 'corpus')
    imgs = sorted(glob.glob('corpus/images/**/*.jpg', recursive=True))
    assert imgs, 'empty corpus — set USE_DRIVE_CORPUS / DRIVE_CORPUS (a .zip or folder on your Drive)'
    print('corpus images on Drive:', len(imgs))
    picks = imgs[:N_IMAGES] if SELECT == 'first' else imgs[::max(1, len(imgs)//N_IMAGES)][:N_IMAGES]
    for p in picks: shutil.copy(p, f'{INP}/{os.path.basename(p)}')
    print('inputs ready:', len(glob.glob(f'{INP}/*')))

## Cell 6 · Run TRELLIS  ->  one 3DGS `.ply` per image  ->  `work/vessels/`
Re-runnable (existing `.ply`s skipped).

In [ ]:
import os, sys, glob
# HF token from Colab secrets (optional — silences the unauth warning / faster DLs; repo is public)
try:
    from google.colab import userdata
    for _n in ('HF_TOKEN', 'HUGGINGFACE_TOKEN', 'HUGGINGFACEHUB_API_TOKEN'):
        try:
            _t = userdata.get(_n)
            if _t: os.environ['HF_TOKEN'] = os.environ['HUGGING_FACE_HUB_TOKEN'] = _t; print('HF token set from', _n); break
        except Exception: pass
except Exception: pass

os.chdir('/content/MAT201B_Projects/reagency/factory')
try: STEPS
except NameError: STEPS = 50
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO']  = 'native'
from PIL import Image
sys.path.insert(0, '/content/TRELLIS')
from trellis.pipelines import TrellisImageTo3DPipeline

pipe = TrellisImageTo3DPipeline.from_pretrained('JeffreyXiang/TRELLIS-image-large'); pipe.cuda()
imgs = sorted(glob.glob('work/vessel_inputs/*.jpg') + glob.glob('work/vessel_inputs/*.png'))
print(f'{len(imgs)} images -> splats (STEPS={STEPS})')
for i, ip in enumerate(imgs):
    name = os.path.splitext(os.path.basename(ip))[0]; out = f'work/vessels/{name}.ply'
    if os.path.exists(out) and os.path.getsize(out) > 0:
        continue
    try:
        outputs = pipe.run(Image.open(ip).convert('RGB'), seed=1,
                           sparse_structure_sampler_params={'steps': STEPS, 'cfg_strength': 7.5},
                           slat_sampler_params={'steps': STEPS, 'cfg_strength': 3.0})
        outputs['gaussian'][0].save_ply(out)
        print(f'[{i+1}/{len(imgs)}] {name}.ply  {os.path.getsize(out)/1e6:.1f} MB')
    except Exception as e:
        print(f'[{i+1}/{len(imgs)}] SKIP {name}: {e}')
print('done -> work/vessels/', len(glob.glob('work/vessels/*.ply')), 'plys')

## Cell 7 · Pack  ->  `../assets/vessel.wswv`  (existing Stage-D packer)
Prunes each keyframe to a fixed `G`, normalizes to one global AABB, Morton-orders -> a real morph (not swimming). `--max-mb {MAX_MB}` keeps `G=30000` from being auto-reduced.

In [ ]:
import os
os.chdir('/content/MAT201B_Projects/reagency/factory')
!python3 stage_d_vessel.py --plys work/vessels --G {G} --max-mb {MAX_MB}
print('vessel.wswv:', round(os.path.getsize('../assets/vessel.wswv')/1e6, 2), 'MB')

## Cell 8 · Download to the Mac  (NEVER git-push from Colab)
Then: `cp ~/Downloads/vessel.wswv MAT201B_Projects/reagency/assets/vessel.wswv` — the runtime tries `assets/vessel.wswv` first, so it auto-loads next launch. (`DENS=4` already set in `VesselSplats.cpp`.)

In [ ]:
import os, glob
from google.colab import files
cands = ['/content/MAT201B_Projects/reagency/assets/vessel.wswv']
OUT = next((p for p in cands if os.path.isfile(p)), None)
if OUT is None:
    hits = glob.glob('/content/**/vessel.wswv', recursive=True); OUT = hits[0] if hits else None
assert OUT, "vessel.wswv not found — did Cell 7 run? try: !find / -name vessel.wswv 2>/dev/null"
print('downloading %s (%.1f MB)...' % (OUT, os.path.getsize(OUT)/1e6))
files.download(OUT)
# --- or stage on Drive instead of a browser download: ---
# import shutil; os.makedirs('/content/drive/MyDrive/wosw', exist_ok=True)
# shutil.copy(OUT, '/content/drive/MyDrive/wosw/vessel.wswv'); print('staged on Drive')

## Troubleshooting
- **Cell 1 keeps restarting / loops:** run Cell 1 alone once, let it reconnect, then Run-all. The `condacolab.check()` guard makes it a no-op after the first install.
- **`setup.sh --kaolin` fails:** install the matching wheel explicitly — `pip install kaolin==0.15.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.1.0_cu118.html` (adjust kaolin version if needed), then re-run Cell 4.
- **`from_pretrained` errors about a missing renderer (nvdiffrast / diffoctreerast):** add `--nvdiffrast --diffoctreerast` to the `setup.sh` line in Cell 4 and re-run it.
- **OOM at `pipe.cuda()` / run:** you're not on an A100 — switch the runtime type, or lower `STEPS`.
- **Want fewer/cheaper keyframes:** lower `N_IMAGES` (and re-run from Cell 5).